# Session 3: the Hodgkin-Huxley model
**A. Real-world problem.** The leaky integrate-and-fire neuron of Session 2 *draws* a spike
by hand. A real nerve cell makes its spike from the physics of ion channels: sodium rushes
in, potassium flows out. Hodgkin and Huxley (squid giant axon, 1952) wrote four coupled
equations that reproduce the spike, the threshold, the refractory period and repetitive
firing. Today you integrate them yourself and validate the result.

**Plan.** (1) write the equations, (2) integrate with RK4, (3) check the resting state,
the gating variables and the charge balance, (4) find the threshold and the firing
onset, (5) test temperature, g_Na, axon diameter and a time step that is too large.

*General education only. This is a simplified textbook model of one axon, not a medical tool.*

In [ ]:
import math
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

# Classic squid-axon parameters, MODERN convention: V = inside minus outside, rest near -65 mV.
C_M = 1.0          # membrane capacitance, uF/cm^2
G_NA = 120.0       # maximum sodium conductance, mS/cm^2
G_K = 36.0         # maximum potassium conductance, mS/cm^2
G_L = 0.3          # leak conductance, mS/cm^2
E_NA = 50.0        # sodium reversal potential, mV
E_K = -77.0        # potassium reversal potential, mV
E_L = -54.387      # leak reversal potential, mV (chosen so that rest is -65 mV)
T_REF = 6.3        # temperature of the original rate functions, degrees C
Q10 = 3.0          # rate multiplier per 10 degrees C (the Q10 idea)

## Step 1: the equations
The membrane voltage V obeys a current balance (units: uA/cm^2 = mS/cm^2 x mV):

    C dV/dt = I_ext - g_Na m^3 h (V - E_Na) - g_K n^4 (V - E_K) - g_L (V - E_L)

Each gate x in {m, n, h} is a probability between 0 and 1 that relaxes towards a value set
by the voltage:  dx/dt = alpha_x(V) (1 - x) - beta_x(V) x.  The rates alpha, beta
(in 1/ms) are the standard empirical fits of Hodgkin and Huxley.

In [ ]:
def vtrap(x, y):
    """x / (1 - exp(-x / y)), safe at x = 0 where the limit is y."""
    z = x / y
    if abs(z) < 1e-7:
        return y * (1.0 - z / 2.0)
    return x / (1.0 - math.exp(-z))


def rates(V):
    """Return (alpha_m, beta_m, alpha_h, beta_h, alpha_n, beta_n) in 1/ms at 6.3 C."""
    am = 0.1 * vtrap(V + 40.0, 10.0)
    bm = 4.0 * math.exp(-(V + 65.0) / 18.0)
    ah = 0.07 * math.exp(-(V + 65.0) / 20.0)
    bh = 1.0 / (1.0 + math.exp(-(V + 35.0) / 10.0))
    an = 0.01 * vtrap(V + 55.0, 10.0)
    bn = 0.125 * math.exp(-(V + 65.0) / 80.0)
    return am, bm, ah, bh, an, bn


def gate_steady(V):
    """Steady-state values (m_inf, h_inf, n_inf) of the gates at a fixed voltage."""
    am, bm, ah, bh, an, bn = rates(V)
    return am / (am + bm), ah / (ah + bh), an / (an + bn)


def ion_currents(V, m, h, n, gna=G_NA):
    """Sodium, potassium and leak currents (uA/cm^2); positive means outward."""
    return gna * m ** 3 * h * (V - E_NA), G_K * n ** 4 * (V - E_K), G_L * (V - E_L)


def hh_rhs(y, i_ext, phi=1.0, gna=G_NA):
    """Right-hand side of the four equations. y = (V, m, h, n); phi multiplies the gate rates."""
    V, m, h, n = y
    am, bm, ah, bh, an, bn = rates(V)
    i_na, i_k, i_l = ion_currents(V, m, h, n, gna)
    return (
        (i_ext - i_na - i_k - i_l) / C_M,
        phi * (am * (1.0 - m) - bm * m),
        phi * (ah * (1.0 - h) - bh * h),
        phi * (an * (1.0 - n) - bn * n),
    )

## Step 2: RK4, written by hand
Runge-Kutta 4 takes four slope estimates per step and combines them. The stimulus
`stim(t)` is a function of time (uA/cm^2), so we can use pulses, steps and ramps.

In [ ]:
def rk4_step(y, t, dt, stim, phi=1.0, gna=G_NA):
    k1 = hh_rhs(y, stim(t), phi, gna)
    y2 = tuple(a + 0.5 * dt * b for a, b in zip(y, k1))
    k2 = hh_rhs(y2, stim(t + 0.5 * dt), phi, gna)
    y3 = tuple(a + 0.5 * dt * b for a, b in zip(y, k2))
    k3 = hh_rhs(y3, stim(t + 0.5 * dt), phi, gna)
    y4 = tuple(a + dt * b for a, b in zip(y, k3))
    k4 = hh_rhs(y4, stim(t + dt), phi, gna)
    return tuple(a + dt / 6.0 * (b + 2 * c + 2 * d + e) for a, b, c, d, e in zip(y, k1, k2, k3, k4))


def find_rest(gna=G_NA):
    """Resting voltage: where the steady-state net ion current is zero (bisection)."""
    def net(V):
        m, h, n = gate_steady(V)
        return sum(ion_currents(V, m, h, n, gna))
    lo, hi = -80.0, -50.0
    for _ in range(80):
        mid = 0.5 * (lo + hi)
        if net(lo) * net(mid) <= 0:
            hi = mid
        else:
            lo = mid
    return 0.5 * (lo + hi)


V_REST = find_rest()


def rest_state(gna=G_NA):
    v = V_REST if gna == G_NA else find_rest(gna)
    m, h, n = gate_steady(v)
    return (v, m, h, n)


def simulate(stim, t_end, dt=0.01, temp=T_REF, gna=G_NA, y0=None):
    """Integrate with RK4 from rest. Returns arrays t, V, m, h, n (ms, mV, then 0..1)."""
    phi = Q10 ** ((temp - T_REF) / 10.0)
    steps = int(round(t_end / dt))
    y = rest_state(gna) if y0 is None else tuple(y0)
    out = np.empty((steps + 1, 4))
    out[0] = y
    for i in range(steps):
        try:
            y = rk4_step(y, i * dt, dt, stim, phi, gna)
        except OverflowError:               # exp() overflowed: the solution has blown up
            out[i + 1:] = np.nan
            break
        out[i + 1] = y
        if not (abs(y[0]) < 1e4):          # blown up: stop and keep what we have
            out[i + 2:] = np.nan
            break
    t = np.arange(steps + 1) * dt
    return t, out[:, 0], out[:, 1], out[:, 2], out[:, 3]


def pulse(amp, start=1.0, dur=1.0):
    return lambda t: amp if start <= t < start + dur else 0.0


def step_current(amp, start=1.0):
    return lambda t: amp if t >= start else 0.0

## Step 3: spikes, thresholds and firing rate
A *spike* is counted each time V crosses 0 mV going upwards. The *threshold* is found by
bisection: the smallest pulse that still produces a spike.

In [ ]:
def spike_times(t, V, level=0.0):
    up = np.where((V[:-1] < level) & (V[1:] >= level))[0]
    return t[up] + (level - V[up]) / (V[up + 1] - V[up]) * (t[up + 1] - t[up])


def fires(amp, dur=1.0, dt=0.02, temp=T_REF, gna=G_NA):
    t, V, *_ = simulate(pulse(amp, 1.0, dur), 30.0 + dur, dt, temp, gna)
    return len(spike_times(t, V)) > 0


def threshold_pulse(dur=1.0, dt=0.02, temp=T_REF, gna=G_NA, hi=400.0):
    """Smallest pulse amplitude (uA/cm^2, to 0.01) of duration `dur` ms that fires a spike."""
    lo = 0.0
    if not fires(hi, dur, dt, temp, gna):
        return float("nan")
    while hi - lo > 0.01:
        mid = 0.5 * (lo + hi)
        if fires(mid, dur, dt, temp, gna):
            hi = mid
        else:
            lo = mid
    return hi


def firing_rate(i_amp, t_end=400.0, dt=0.05, temp=T_REF, gna=G_NA):
    """Steady rate (Hz) for a constant current: from the last inter-spike interval."""
    t, V, *_ = simulate(step_current(i_amp, 1.0), t_end, dt, temp, gna)
    st = spike_times(t, V)
    if len(st) < 3:
        return 0.0
    return 1000.0 / float(st[-1] - st[-2])


def strength_duration(durations=(0.2, 0.5, 1.0, 2.0, 5.0, 20.0)):
    """Mini-project helper: threshold amplitude (uA/cm^2) and charge (nC/cm^2) for each pulse length."""
    return [(d, threshold_pulse(d), d * threshold_pulse(d)) for d in durations]


def max_rate_scan(currents):
    """Mini-project helper: (best rate in Hz, current) over a list of constant currents."""
    best = (0.0, None)
    for i in currents:
        r = firing_rate(float(i))
        if r > best[0]:
            best = (r, float(i))
    return best


def onset_current(lo=2.0, hi=12.0, t_end=1000.0, dt=0.05):
    """Smallest constant current (to 0.01 uA/cm^2) with sustained firing: a spike still occurs in
    the last fifth of a `t_end` ms run (a few transient spikes at the start do not count)."""
    def sustained(i):
        t, V, *_ = simulate(step_current(i, 1.0), t_end, dt)
        return bool(np.any(spike_times(t, V) > 0.8 * t_end))
    while hi - lo > 0.01:
        mid = 0.5 * (lo + hi)
        if sustained(mid):
            hi = mid
        else:
            lo = mid
    return hi

## Step 4: validation against a reference
For a tiny current the gates barely move in the first millisecond, so the membrane acts
as one leaky capacitor with a fixed conductance g0 = g_Na m^3 h + g_K n^4 + g_L. That is
the leaky integrate-and-fire closed form of Session 2:
V(t) = V_rest + (I / g0) (1 - exp(-g0 t / C)).

In [ ]:
def instant_conductance():
    m, h, n = gate_steady(V_REST)
    return G_NA * m ** 3 * h + G_K * n ** 4 + G_L


def lif_subthreshold(t, i_amp, g0=None):
    g0 = instant_conductance() if g0 is None else g0
    return V_REST + (i_amp / g0) * (1.0 - np.exp(-g0 * t / C_M))


def charge_balance(amp=10.0, dur=1.0, t_end=40.0, dt=0.01):
    """Charges (uA/cm^2 x ms = nC/cm^2) carried by each ion current during one spike.
    Returns (q_na, q_k, q_l, q_stim, delta_v). The stimulus charge must equal the net ion
    charge plus C x change in V."""
    t, V, m, h, n = simulate(pulse(amp, 1.0, dur), t_end, dt)
    i_na = G_NA * m ** 3 * h * (V - E_NA)
    i_k = G_K * n ** 4 * (V - E_K)
    i_l = G_L * (V - E_L)
    q = [float(np.sum(0.5 * (x[1:] + x[:-1]) * np.diff(t))) for x in (i_na, i_k, i_l)]
    return q[0], q[1], q[2], amp * dur, float(V[-1] - V[0])

## Step 5: temperature, g_Na, a refractory period and a step that is too large

In [ ]:
def spike_width(t, V):
    """Width (ms) of the first spike at half of its height above rest."""
    k = int(np.argmax(V))
    half = V_REST + 0.5 * (V[k] - V_REST)
    a = k
    while a > 0 and V[a] > half:
        a -= 1
    b = k
    while b < len(V) - 1 and V[b] > half:
        b += 1
    ta = t[a] + (half - V[a]) / (V[a + 1] - V[a]) * (t[a + 1] - t[a])
    tb = t[b - 1] + (half - V[b - 1]) / (V[b] - V[b - 1]) * (t[b] - t[b - 1])
    return float(tb - ta)


def second_pulse_threshold(gap, first=20.0, dur=1.0, dt=0.02):
    """Smallest amplitude of a second 1 ms pulse that fires a second spike `gap` ms after the
    start of the first pulse (which is 1 ms long and strong enough to fire). nan if none."""
    def stim_of(a):
        return lambda t: first if 1.0 <= t < 1.0 + dur else (a if 1.0 + gap <= t < 1.0 + gap + dur else 0.0)

    def n_spikes(a):
        t, V, *_ = simulate(stim_of(a), 40.0, dt)
        return len(spike_times(t, V))

    hi = 400.0
    if n_spikes(hi) < 2:
        return float("nan")
    lo = 0.0
    while hi - lo > 0.05:
        mid = 0.5 * (lo + hi)
        if n_spikes(mid) >= 2:
            hi = mid
        else:
            lo = mid
    return hi


def blows_up(dt, i_amp=10.0, t_end=20.0):
    """True if RK4 with this time step leaves the physical range (V beyond +-200 mV or gate outside 0..1)."""
    t, V, m, h, n = simulate(pulse(i_amp, 1.0, 1.0), t_end, dt)
    bad = ~np.isfinite(V) | (np.abs(V) > 200.0)
    for g in (m, h, n):
        bad = bad | ~np.isfinite(g) | (g < -1e-9) | (g > 1 + 1e-9)
    return bool(bad.any())

## Step 6: how fast does the spike travel?
Cable theory: a spike spreads along the axon by current flowing through the inside (axial
resistance r_i per length) and leaking out through the membrane (r_m x length). The length
constant is lambda = sqrt(r_m / r_i). For an axon of radius a, r_i is proportional to
1/a^2 and r_m to 1/a, so lambda is proportional to sqrt(a). The membrane time constant does
not depend on a, so the speed (about lambda / time) scales as **sqrt(a)**. We test this
scaling with a discretised HH cable.

In [ ]:
RI = 35.4          # axoplasm resistivity, ohm cm (classic squid value)


def cable_speed(radius_cm, length_cm=6.0, dx=0.02, dt=0.0005, t_max=40.0):
    """Speed (cm/ms) of a spike travelling on an HH cable of the given radius (explicit scheme)."""
    with np.errstate(all="ignore"):
        return _cable_speed(radius_cm, length_cm, dx, dt, t_max)


def _cable_speed(radius_cm, length_cm, dx, dt, t_max):
    n = int(round(length_cm / dx)) + 1
    v0, m0, h0, n0 = rest_state()
    V = np.full(n, v0)
    m = np.full(n, m0)
    h = np.full(n, h0)
    g = np.full(n, n0)
    d = 1000.0 * radius_cm / (2.0 * RI * C_M)     # cm^2/ms (the factor 1000 converts mA to uA)
    x = np.arange(n) * dx
    t_a = t_b = None
    xa, xb = 1.5, 4.0
    ia, ib = int(round(xa / dx)), int(round(xb / dx))
    for step in range(int(t_max / dt)):
        t = step * dt
        lap = np.empty(n)
        lap[1:-1] = (V[2:] - 2 * V[1:-1] + V[:-2]) / dx ** 2
        lap[0] = 2 * (V[1] - V[0]) / dx ** 2
        lap[-1] = 2 * (V[-2] - V[-1]) / dx ** 2
        am = 0.1 * (V + 40.0) / (1.0 - np.exp(-(V + 40.0) / 10.0))
        bm = 4.0 * np.exp(-(V + 65.0) / 18.0)
        ah = 0.07 * np.exp(-(V + 65.0) / 20.0)
        bh = 1.0 / (1.0 + np.exp(-(V + 35.0) / 10.0))
        an = 0.01 * (V + 55.0) / (1.0 - np.exp(-(V + 55.0) / 10.0))
        bn = 0.125 * np.exp(-(V + 65.0) / 80.0)
        i_ion = G_NA * m ** 3 * h * (V - E_NA) + G_K * g ** 4 * (V - E_K) + G_L * (V - E_L)
        i_inj = np.where((x < 0.2) & (t < 1.0), 40.0, 0.0)
        dV = d * lap + (i_inj - i_ion) / C_M
        m += dt * (am * (1 - m) - bm * m)
        h += dt * (ah * (1 - h) - bh * h)
        g += dt * (an * (1 - g) - bn * g)
        V += dt * dV
        if t_a is None and V[ia] > 0.0:
            t_a = t
        if t_b is None and V[ib] > 0.0:
            t_b = t
            break
    if t_a is None or t_b is None:
        return float("nan")
    return (xb - xa) / (t_b - t_a)


def velocity_scaling(a, a0, v0):
    """Scaling argument: v is proportional to sqrt(radius). Returns v at radius a given v0 at a0."""
    return v0 * math.sqrt(a / a0)

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    print(f"resting potential {V_REST:.3f} mV; gates at rest m,h,n = "
          + ", ".join(f"{g:.4f}" for g in gate_steady(V_REST)))

    t, V, m, h, n = simulate(pulse(10.0, 1.0, 1.0), 30.0, 0.01)
    st = spike_times(t, V)
    print(f"10 uA/cm2 for 1 ms: peak {V.max():.1f} mV at {t[V.argmax()]:.2f} ms, "
          f"width {spike_width(t, V):.2f} ms, undershoot {V[t > 5].min():.1f} mV, spikes {len(st)}")
    fig, ax = plt.subplots(2, 1, figsize=(7, 6), sharex=True)
    ax[0].plot(t, V, color="#1f6feb")
    ax[0].axhline(V_REST, color="gray", ls=":")
    ax[0].set_ylabel("membrane voltage V (mV)")
    ax[0].set_title("One action potential from a 1 ms, 10 uA/cm^2 pulse")
    ax[1].plot(t, m, label="m (Na activation)")
    ax[1].plot(t, h, label="h (Na inactivation)")
    ax[1].plot(t, n, label="n (K activation)")
    ax[1].set_xlabel("time (ms)")
    ax[1].set_ylabel("gate value (0 to 1)")
    ax[1].legend(fontsize=8)
    for a in ax:
        a.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(OUT / "action_potential.png", dpi=120)

    # threshold behaviour
    thr = threshold_pulse(1.0)
    print(f"threshold for a 1 ms pulse: {thr:.2f} uA/cm2")
    fig2, ax2 = plt.subplots(figsize=(7, 4))
    for amp in (0.8 * thr, 0.95 * thr, 1.05 * thr, 2 * thr):
        tt, VV, *_ = simulate(pulse(amp, 1.0, 1.0), 20.0, 0.02)
        ax2.plot(tt, VV, label=f"{amp:.1f} uA/cm2")
        print(f"  pulse {amp:6.2f}: peak {VV.max():6.1f} mV")
    ax2.set_xlabel("time (ms)")
    ax2.set_ylabel("membrane voltage V (mV)")
    ax2.set_title("All or nothing: weak pulses fade, strong ones give the same spike")
    ax2.legend(fontsize=8)
    ax2.grid(alpha=0.3)
    fig2.tight_layout()
    fig2.savefig(OUT / "threshold.png", dpi=120)

    # repetitive firing and f-I curve
    cur = [2, 4, 6, 8, 10, 15, 20, 30, 50, 100]
    rates_hz = [firing_rate(i) for i in cur]
    onset = onset_current()
    print(f"onset of sustained firing: {onset:.2f} uA/cm2")
    for i, r in zip(cur, rates_hz):
        print(f"  I = {i:4d} uA/cm2 -> {r:6.1f} Hz")
    fig3, ax3 = plt.subplots(figsize=(7, 4))
    ax3.plot(cur, rates_hz, "o-")
    ax3.axvline(onset, color="red", ls="--", label=f"measured onset {onset:.1f}")
    ax3.set_xlabel("constant current I (uA/cm^2)")
    ax3.set_ylabel("firing rate (spikes per second)")
    ax3.set_title("Hodgkin-Huxley f-I curve")
    ax3.legend(fontsize=8)
    ax3.grid(alpha=0.3)
    fig3.tight_layout()
    fig3.savefig(OUT / "fi_curve.png", dpi=120)

    # validation
    tt = np.arange(0, 1.001, 0.01)
    t2, V2, *_ = simulate(step_current(0.1, 0.0), 1.0, 0.01)
    print(f"LIF check at 1 ms for 0.1 uA/cm2: HH {V2[-1] - V_REST:.5f} mV, LIF {lif_subthreshold(tt, 0.1)[-1] - V_REST:.5f} mV")
    print("charge balance (na, k, leak, stim, dV):", [round(x, 4) for x in charge_balance()])

    # a time step that is too large
    for dt in (0.01, 0.1, 0.5, 1.0, 2.0, 3.0):
        print(f"  dt = {dt}: blows up = {blows_up(dt)}")

## Coding tasks
1. **Easy.** Run `simulate` with `pulse(10.0, 1.0, 1.0)` for dt = 0.05 and 0.01 and print how much the peak changes; then find the largest dt (to 0.01) at which `blows_up` stays False.
2. **Stretch.** Write `refractory_curve(gaps)` that prints `second_pulse_threshold` for gaps of 5, 8, 10 and 15 ms and plot the result: how does the needed amplitude fall with the gap?
3. **Explain.** Explain, with the gating variables of the figure, why the cell cannot fire again 4 ms after the start of a pulse even with an enormous second pulse.

## Assessment
- **A. Conceptual.** Why does a spike appear suddenly once the stimulus passes a threshold, and why is it the same size for every stronger stimulus?
- **B. Numerical.** For a tiny current of 0.1 uA/cm^2 the membrane acts, for the first moments, like a leaky capacitor with g_0 = 0.6775 mS/cm^2. Calculate by hand the voltage rise after 0.2 ms and compare it with the simulation.
- **C. Graph and simulation interpretation.** On the f-I curve, what happens at currents below 6.27, just above it, and at very large currents? What does this say about the cell?
- **D. Design and optimisation challenge.** Design: a spike must travel 1 m in 20 ms, that is 50 m/s. Using the scaling v proportional to sqrt(a) and the simulated 7.97 m/s for a 100 um radius, what radius would an unmyelinated axon need? What does this suggest, and what temperature effect would you expect?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import math

import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    # resting state
    checks.append((f"resting potential is -65.0 mV (found {L.V_REST:.3f})", abs(L.V_REST + 65.0) < 0.01))
    m0, h0, n0 = L.gate_steady(L.V_REST)
    checks.append(("gates at rest: m = 0.053, h = 0.596, n = 0.318", abs(m0 - 0.0530) < 5e-4 and abs(h0 - 0.5960) < 5e-4 and abs(n0 - 0.3177) < 5e-4))
    t, V, m, h, n = L.simulate(lambda t: 0.0, 50.0, 0.01)
    checks.append(("validation 1: with no stimulus the voltage stays at rest to 1e-3 mV for 50 ms", float(np.max(np.abs(V - L.V_REST))) < 1e-3))
    am, bm, ah, bh, an, bn = L.rates(-40.0)
    checks.append(("edge case: rate alpha_m is finite at V = -40 mV where 0/0 appears (value 1.0 per ms)", abs(am - 1.0) < 1e-6))
    checks.append(("edge case: rate alpha_n is finite at V = -55 mV (value 0.1 per ms)", abs(L.rates(-55.0)[4] - 0.1) < 1e-6))

    # one action potential
    t, V, m, h, n = L.simulate(L.pulse(10.0, 1.0, 1.0), 30.0, 0.01)
    st = L.spike_times(t, V)
    checks.append((f"worked example: a 10 uA/cm2, 1 ms pulse gives one spike peaking at 39.1 mV (got {V.max():.1f})", len(st) == 1 and abs(V.max() - 39.1) < 0.1))
    checks.append(("the spike peaks 3.5 ms after the start and undershoots to -76.2 mV (below rest, near E_K = -77)",
                   abs(t[V.argmax()] - 3.51) < 0.02 and abs(V[t > 5].min() + 76.2) < 0.1))
    checks.append(("spike width at half height is 1.45 ms at 6.3 C", abs(L.spike_width(t, V) - 1.45) < 0.02))
    checks.append(("peak stays between E_K and E_Na (the voltage cannot pass the reversal potentials)", float(V.min()) > L.E_K and float(V.max()) < L.E_NA))
    # gating variables
    ok = True
    for g in (m, h, n):
        ok &= bool(g.min() >= 0.0 and g.max() <= 1.0)
    checks.append(("conservation-like check 1: m, h, n stay inside [0, 1] during a spike", ok))
    checks.append(("m peaks at 0.994 (fast), h dips to 0.076, n reaches 0.768", abs(m.max() - 0.9937) < 5e-4 and abs(h.min() - 0.0764) < 5e-4 and abs(n.max() - 0.7681) < 5e-4))
    # numerical convergence
    t2, V2, *_ = L.simulate(L.pulse(10.0, 1.0, 1.0), 30.0, 0.005)
    checks.append(("validation 2: halving dt to 0.005 ms changes the peak by less than 0.01 mV", abs(V2.max() - V.max()) < 0.01))
    t3, V3, *_ = L.simulate(L.pulse(10.0, 1.0, 1.0), 30.0, 0.05)
    checks.append(("RK4 with dt = 0.05 ms is still within 0.2 mV on the peak", abs(V3.max() - V.max()) < 0.2))
    # charge balance
    qna, qk, ql, qs, dv = L.charge_balance()
    checks.append((f"conservation-like check 2: Na carries {qna:.0f} (inward), K {qk:.0f}, leak {ql:.0f} nC/cm2; their sum equals the 10 nC/cm2 stimulus",
                   qna < 0 < qk and abs(qna + qk + ql - qs + L.C_M * dv) < 0.05 and abs(qna + qk + ql - 10.0) < 0.1))
    checks.append(("Na inward charge is about 1427 nC/cm2, K outward 1535 nC/cm2", abs(qna + 1426.6) < 1.0 and abs(qk - 1535.5) < 1.0))
    checks.append(("after the spike the voltage is back within 0.02 mV of rest at 40 ms", abs(dv) < 0.02))

    # LIF closed form for the subthreshold response
    g0 = L.instant_conductance()
    checks.append(("instantaneous conductance at rest is 0.678 mS/cm2, so tau = C/g = 1.476 ms", abs(g0 - 0.6775) < 5e-4 and abs(L.C_M / g0 - 1.476) < 0.002))
    t4, V4, *_ = L.simulate(L.step_current(0.1, 0.0), 0.2, 0.01)
    hh, lif = V4[-1] - L.V_REST, float(L.lif_subthreshold(t4[-1:], 0.1)[0] - L.V_REST)
    checks.append((f"validation 3: after 0.2 ms the HH response ({hh:.5f} mV) matches the LIF closed form ({lif:.5f} mV) within 2 percent", abs(hh / lif - 1) < 0.02))
    t5, V5, *_ = L.simulate(L.step_current(0.1, 0.0), 1.0, 0.01)
    ratio = (V5[-1] - L.V_REST) / float(L.lif_subthreshold(t5[-1:], 0.1)[0] - L.V_REST)
    checks.append((f"by 1 ms the gates have moved and HH is {100 * (ratio - 1):.0f} percent above the LIF curve (the limit of the leaky model)", 1.09 < ratio < 1.13))

    # threshold behaviour
    thr = L.threshold_pulse(1.0)
    checks.append((f"worked example: the threshold of a 1 ms pulse is 6.92 uA/cm2 (found {thr:.2f})", abs(thr - 6.92) < 0.02))
    peaks = []
    for amp in (6.5, 7.5, 15.0, 30.0):
        tt, VV, *_ = L.simulate(L.pulse(amp, 1.0, 1.0), 20.0, 0.02)
        peaks.append(float(VV.max()))
    checks.append((f"all or nothing: 6.5 uA/cm2 peaks at {peaks[0]:.1f} mV (no spike); 7.5, 15 and 30 all peak between 36 and 42 mV",
                   abs(peaks[0] + 59.1) < 0.2 and all(36 < p < 42 for p in peaks[1:])))
    # refractory
    checks.append(("refractory: 4 ms after the first pulse no second spike is possible even with 400 uA/cm2", math.isnan(L.second_pulse_threshold(4.0))))
    s5, s10 = L.second_pulse_threshold(5.0), L.second_pulse_threshold(10.0)
    checks.append((f"relative refractory: the second pulse needs {s5:.0f} uA/cm2 at 5 ms and {s10:.1f} at 10 ms, far above the threshold 6.92",
                   abs(s5 - 275.1) < 1.0 and abs(s10 - 23.6) < 0.3 and s10 > 3 * thr))
    # repetitive firing
    onset = L.onset_current()
    checks.append((f"worked example: sustained firing starts at {onset:.2f} uA/cm2 (literature: about 6)", abs(onset - 6.27) < 0.02))
    t6, V6, *_ = L.simulate(L.step_current(5.0, 1.0), 300.0, 0.05)
    checks.append(("between the rheobase and the onset (5 uA/cm2) the cell fires exactly once and then rests", len(L.spike_times(t6, V6)) == 1 and L.firing_rate(5.0) == 0.0))
    f8, f10 = L.firing_rate(8.0), L.firing_rate(10.0)
    checks.append((f"just above onset the rate jumps to about 60 Hz (8 uA/cm2: {f8:.1f}, 10: {f10:.1f})", abs(f8 - 62.5) < 0.5 and abs(f10 - 68.3) < 0.5))
    rates = {i: L.firing_rate(float(i)) for i in range(10, 66, 5)}
    best_i = max(rates, key=rates.get)
    checks.append((f"maximum firing rate: about 126 Hz near 60 to 65 uA/cm2 (scan best {rates[best_i]:.1f} Hz at {best_i})", 122 < max(rates.values()) < 128 and best_i in (60, 65)))
    checks.append(("depolarisation block: at 66 uA/cm2 the cell stops firing repetitively", L.firing_rate(66.0) == 0.0))

    # mini-project helpers
    sd = L.strength_duration((0.2, 1.0, 50.0))
    checks.append((f"mini-project: lowest current for a long pulse (rheobase) is {sd[2][1]:.2f} uA/cm2; 1 ms needs 6.92; 0.2 ms needs 32.66",
                   abs(sd[2][1] - 2.25) < 0.02 and abs(sd[1][1] - 6.92) < 0.02 and abs(sd[0][1] - 32.66) < 0.05))
    checks.append(("mini-project: the smallest charge is for the shortest pulse (6.53 nC/cm2 at 0.2 ms, 6.92 at 1 ms, 112 at 50 ms)",
                   abs(sd[0][2] - 6.53) < 0.02 and sd[0][2] < sd[1][2] < sd[2][2] and abs(sd[2][2] - 112.3) < 0.5))
    best = L.max_rate_scan(range(55, 66))
    checks.append((f"mini-project: scanning 55 to 65 uA/cm2 the highest rate is {best[0]:.1f} Hz at {best[1]:.0f} uA/cm2", abs(best[0] - 125.87) < 0.1 and best[1] == 63.0))

    # temperature and g_Na
    thr18 = L.threshold_pulse(1.0, temp=18.5)
    tt, VV, *_ = L.simulate(L.pulse(20.0, 1.0, 1.0), 30.0, 0.01, temp=18.5)
    w18 = L.spike_width(tt, VV)
    checks.append((f"Q10: at 18.5 C the spike is {w18:.2f} ms wide (1.45 at 6.3 C) and the 1 ms threshold rises to {thr18:.2f}", abs(w18 - 0.42) < 0.02 and abs(thr18 - 8.91) < 0.03))
    r18 = L.firing_rate(20.0, temp=18.5, t_end=200.0)
    checks.append((f"Q10: at 20 uA/cm2 the rate goes from 86.5 Hz (6.3 C) to {r18:.0f} Hz (18.5 C)", abs(L.firing_rate(20.0, t_end=200.0) - 86.5) < 0.5 and abs(r18 - 254.0) < 2.0))
    checks.append(("the rate factor is 3^1.22 = 3.8 between 6.3 and 18.5 C", abs(L.Q10 ** ((18.5 - 6.3) / 10) - 3.82) < 0.01))
    th = {g: L.threshold_pulse(1.0, gna=g) for g in (120.0, 100.0, 80.0, 40.0)}
    checks.append(("g_Na sensitivity: the threshold rises as g_Na falls (120: 6.92, 100: 8.56, 80: 10.61, 40: 17.86 uA/cm2)",
                   th[120.0] < th[100.0] < th[80.0] < th[40.0] and abs(th[100.0] - 8.56) < 0.03 and abs(th[80.0] - 10.61) < 0.03 and abs(th[40.0] - 17.86) < 0.05))
    tg, Vg, *_ = L.simulate(L.pulse(30.0, 1.0, 1.0), 30.0, 0.01, gna=20.0)
    checks.append(("with g_Na = 20 mS/cm2 even a 30 uA/cm2 pulse gives no spike (peak -16.7 mV)", len(L.spike_times(tg, Vg)) == 0 and abs(Vg.max() + 16.7) < 0.3))

    # a step that is too large
    checks.append(("failure case: RK4 with dt = 0.05 ms is fine but dt = 0.1 ms blows up (voltage beyond 200 mV or gates outside 0..1)", not L.blows_up(0.05) and L.blows_up(0.1)))
    checks.append(("failure case: larger steps 0.5, 1 and 2 ms also fail", all(L.blows_up(d) for d in (0.5, 1.0, 2.0))))

    # conduction speed
    v = [10.0 * L.cable_speed(a) for a in (0.005, 0.01, 0.02)]
    checks.append((f"cable simulation: speeds {v[0]:.2f}, {v[1]:.2f}, {v[2]:.2f} m/s for radii 50, 100, 200 um (6.3 C)", abs(v[0] - 5.64) < 0.05 and abs(v[1] - 7.97) < 0.05 and abs(v[2] - 11.30) < 0.05))
    checks.append((f"scaling: doubling the radius multiplies the speed by {v[1] / v[0]:.3f} and {v[2] / v[1]:.3f}, close to sqrt 2 = 1.414", abs(v[1] / v[0] - 1.414) < 0.02 and abs(v[2] / v[1] - 1.414) < 0.02))
    checks.append(("scaling formula: 4 times the radius gives 2 times the speed", abs(L.velocity_scaling(4.0, 1.0, 5.64) - 11.28) < 1e-9))
    T = 1000.0
    ta, Va, *_ = L.simulate(lambda t: 20.0 * t / T, T, 0.05)
    sta = L.spike_times(ta, Va)
    checks.append((f"accommodation: a slow ramp 0 to 20 uA/cm2 over 1000 ms has its first spike at {20 * sta[0] / T:.2f} uA/cm2, far above the steady onset 6.27", len(sta) > 0 and abs(20 * sta[0] / T - 18.06) < 0.1))
    T2 = 2000.0
    tri = lambda t: 30.0 * t / (T2 / 2) if t < T2 / 2 else 30.0 * (T2 - t) / (T2 / 2)
    tb, Vb, *_ = L.simulate(tri, T2, 0.05)
    stb = L.spike_times(tb, Vb)
    cb = [tri(x) for x in stb]
    checks.append((f"hysteresis: on a triangle ramp 0-30-0 uA/cm2 over 2000 ms the first spike is at {cb[0]:.2f} on the way up and the last at {cb[-1]:.2f} on the way down",
                   abs(cb[0] - 19.79) < 0.3 and abs(cb[-1] - 6.88) < 0.3))
    checks.append(("charge of the 1 ms threshold pulse is 6.92 nC/cm2, which raises V by about 6.9 mV on a 1 uF/cm2 membrane", abs(thr * 1.0 / L.C_M - 6.92) < 0.02))
    hand = 0.1 / g0 * (1 - math.exp(-g0 * 0.2))
    checks.append(("question B: hand calculation (0.1 / 0.6775) x (1 - exp(-0.1355)) = 0.1476 x 0.1267 = 0.01870 mV",
                   abs(0.1 / g0 - 0.1476) < 5e-4 and abs(1 - math.exp(-g0 * 0.2) - 0.1267) < 5e-4 and abs(hand - 0.01870) < 5e-5))
    radius = 100.0 * (50.0 / v[1]) ** 2
    checks.append((f"question D: reaching 50 m/s by diameter alone needs a radius of about {radius:.0f} um (scaling from 7.97 m/s at 100 um)", abs(radius - 3936.0) < 20.0))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
